# SEM Analysis Workflow

**Updated:** 2026-01-18

This notebook demonstrates the `SEMAnalyser` framework for processing Scanning Electron Microscopy images.
The module supports automatic metadata extraction from **Zeiss** and **FEI/Thermo Fisher** instruments via HyperSpy.

## Features
- Auto-detect pixel scale via HyperSpy (Zeiss CZ_SEM, FEI metadata)
- Auto-detect and crop footer/data bar
- Aspect ratio adjustment for publication-ready images
- Contrast enhancement (percentile stretch, CLAHE)
- Publication-ready scale bars with auto unit selection (nm/µm)
- Export to multiple formats (TIFF, HyperSpy, NetCDF)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

# 1. Setup Environment
sys.path.append(str(Path.cwd().parent.parent))

try:
    from Figure.config import DATA_ROOT, OUTPUT_ROOT, setup
    colors = setup()
except ImportError:
    print("Using local configuration")
    OUTPUT_ROOT = Path.cwd() / "output"

# 2. Import SEM Module
from Characterization.SEM.sem_utils import SEMAnalyser

%matplotlib inline

## 1. Metadata Extraction

Test automatic metadata extraction from Zeiss and FEI images.

In [ ]:
# Test metadata extraction from test images using SEMAnalyser
test_files = ["SEM.tif", "SEM2.tif"]

for f in test_files:
    if Path(f).exists():
        analyser = SEMAnalyser(f)
        meta = analyser.metadata
        print(f"{f}:")
        print(f"  Source: {meta.source}")
        print(f"  Scale: {meta.pixel_scale:.4f} {meta.units}/pixel" if meta.is_valid() else "  Scale: Not detected")

## 2. Basic Workflow

Process SEM images with automatic metadata extraction and scale bar generation.

In [ ]:
# Process Zeiss SEM Image
analyser = SEMAnalyser("SEM.tif")

print(f"Image: {analyser.metadata.original_filename}")
print(f"Source: {analyser.metadata.source}")
print(f"Scale: {analyser.metadata.pixel_scale:.2f} {analyser.metadata.units}/px")

# Preprocess and plot
analyser.preprocess(crop_bottom=0, contrast_method="stretch")
analyser.plot(scale_bar_size=None, scale_bar_color="white", figsize=(5, 4))

## 3. Scale Bar Customization

Advanced scale bar styling options.

In [ ]:
if Path("SEM.tif").exists():
    analyser = SEMAnalyser("SEM.tif")
    analyser.preprocess(contrast_method="clahe")  # Try CLAHE enhancement

    # Custom scale bar with background box
    analyser.plot(
        scale_bar_size=50,        # 50 µm scale bar
        scale_bar_units="µm",
        scale_bar_color="black",
        scale_bar_location="lower right",
        scale_bar_font_size=12,
        scale_bar_frameon=True,
        scale_bar_frame_alpha=0.8,
        scale_bar_frame_color="white",
        scale_bar_height_fraction=0.02,
        figsize=(5, 4)
    )

## 4. Side-by-Side Comparison

Compare multiple images.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, fname, title in zip(axes, ["SEM.tif", "SEM2.tif"], ["Zeiss (SEM.tif)", "FEI (SEM2.tif)"]):
    if Path(fname).exists():
        analyser = SEMAnalyser(fname)
        analyser.preprocess(contrast_method="stretch")

        ax.imshow(analyser.processed_image, cmap="gray")
        ax.set_title(f"{title}\n{analyser.metadata.pixel_scale:.1f} {analyser.metadata.units}/px ({analyser.metadata.source})")
        ax.axis("off")
    else:
        ax.text(0.5, 0.5, f"{fname} not found", ha="center", va="center")
        ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Batch Processing

Process all .tif files and save results.

In [ ]:
# Batch process and save all .tif files
output_dir = OUTPUT_ROOT / "SEM_Processed"
files = list(Path.cwd().glob("*.tif"))

if not files:
    print("No .tif files found.")
else:
    print(f"Processing {len(files)} files -> {output_dir}")
    for f in files:
        try:
            (SEMAnalyser(f)
             .preprocess(contrast_method="stretch")
             .plot(show=False)
             .save(output_dir=output_dir, dpi_list=(300,), save_hyperspy=True, save_xarray=True))
            print(f"  ✓ {f.name}")
        except Exception as e:
            print(f"  ✗ {f.name}: {e}")
    
    # Verify outputs
    print(f"Saved files in {output_dir}:")
    for out_f in sorted(output_dir.glob("*")):
        print(f"  {out_f.name} ({out_f.stat().st_size/1024:.1f} KB)")

## 6. Manual Scale Override

For images without embedded metadata.

In [ ]:
if Path("SEM.tif").exists():
    analyser = SEMAnalyser("SEM.tif")

    # Override with manual scale (e.g., measured from reference)
    analyser.set_scale(pixel_scale=0.25, units="µm")  # 250 nm/pixel

    print(f"Manual scale: {analyser.metadata.pixel_scale} {analyser.metadata.units}")
    print(f"Source: {analyser.metadata.source}")

    analyser.preprocess()
    analyser.plot(
        scale_bar_size=2,  # 2 µm
        scale_bar_units="µm",
        figsize=(5, 4)
    )

## 7. Programmatic Access

Get processed data for further analysis.

In [ ]:
if Path("SEM.tif").exists():
    analyser = SEMAnalyser("SEM.tif")
    analyser.preprocess()

    result = analyser.get_result()

    print("SEMResult:")
    print(f"  Image shape: {result.image.shape}")
    print(f"  Metadata: {result.metadata}")
    print(f"  Processing: {result.processing_params}")